In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver;

In [0]:
from pyspark.sql import functions as F

# Leemos la tabla cruda
df_bronze_sunat = spark.table("bronze.sunat_g6")

# Nuestro parámetro MVP
ANIO_CUADRO = 2005  

MESES = {
    "enero": "01", "febrero": "02", "marzo": "03", "abril": "04",
    "mayo": "05", "junio": "06", "julio": "07", "agosto": "08",
    "setiembre": "09", "octubre": "10", "noviembre": "11", "diciembre": "12",
}

In [0]:
def transformar_sunat_a_largo(df_wide, anio: int):
    columnas_mes = list(MESES.keys())
    stack_expr = ", ".join([f"'{m}', {m}" for m in columnas_mes])
    
    df_largo = df_wide.selectExpr(
        "pais_destino",
        f"stack({len(columnas_mes)}, {stack_expr}) as (mes, valor_fob_usd_miles)"
    )
    
    mapping_expr = F.create_map([F.lit(x) for par in MESES.items() for x in par])
    df_largo = df_largo.withColumn("mes_num", mapping_expr[F.col("mes")])
    
    df_largo = df_largo.withColumn(
        "fecha", F.to_date(F.concat(F.lit(f"{anio}-"), F.col("mes_num"), F.lit("-01")))
    )
    return df_largo.select("pais_destino", "fecha", "valor_fob_usd_miles")

In [0]:
df_sunat_largo = transformar_sunat_a_largo(df_bronze_sunat, ANIO_CUADRO)
print(f"Filas después del unpivot: {df_sunat_largo.count()}") 
display(df_sunat_largo.limit(5))

Filas después del unpivot: 2148


pais_destino,fecha,valor_fob_usd_miles
ESTADOS UNIDOS,2005-01-01,350954745.53
CHINA,2005-01-01,149479103.82
CHILE,2005-01-01,90311287.95
CANADA,2005-01-01,59361869.92
SUIZA,2005-01-01,37051316.41


In [0]:
def limpiar_comercio_exterior(df):
    # Multiplicamos por 1000 y casteamos a double
    df = df.withColumn("valor_fob_usd", (F.col("valor_fob_usd_miles").cast("double") * 1000))
    df = df.dropDuplicates(["fecha", "pais_destino"])
    
    # Regla: descartar nulos, no imputar ceros falsos
    df = df.filter(F.col("valor_fob_usd").isNotNull())
    return df

df_silver_sunat = limpiar_comercio_exterior(df_sunat_largo)
print(f"Filas silver SUNAT: {df_silver_sunat.count()}")

df_silver_sunat.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver.sunat_paises")

Filas silver SUNAT: 2148


In [0]:
df_bronze_comtrade = spark.table("bronze.comtrade")

def limpiar_comtrade(df):
    # 1. Casteamos valores
    df = df.withColumn("valor_fob_usd", F.col("primaryValue").cast("double"))
    df = df.withColumn("anio", F.col("period").cast("int"))
    
    # 2. Rescate del país nulo: Si partnerCode es '0', entonces es el 'MUNDO'
    df = df.withColumn(
        "pais_destino",
        F.when(F.col("partnerCode") == "0", F.lit("MUNDO"))
         .otherwise(F.col("partnerDesc"))
    )
    
    # 3. Limpieza estándar
    df = df.dropDuplicates(["pais_destino", "anio"])
    df = df.filter(F.col("valor_fob_usd").isNotNull())
    
    return df.select("pais_destino", "anio", "valor_fob_usd")

df_silver_comtrade = limpiar_comtrade(df_bronze_comtrade)

df_silver_comtrade.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver.comtrade")

print(f"Filas silver Comtrade: {df_silver_comtrade.count()}")

Filas silver Comtrade: 1


In [0]:
REGLAS_CALIDAD = {
    "valor_no_negativo": "valor_fob_usd >= 0",
    "pais_valido": "pais_destino IS NOT NULL",
}

def generar_reporte_calidad(df, reglas: dict, nombre_tabla: str):
    filas = []
    total = df.count()
    for nombre_regla, expr in reglas.items():
        cumplen = df.filter(expr).count()
        fallan = total - cumplen
        filas.append((nombre_tabla, nombre_regla, total, cumplen, fallan,
                       round(fallan / total * 100, 2) if total else 0.0))
    return spark.createDataFrame(
        filas, ["tabla", "regla", "total", "cumplen", "fallan", "pct_fallan"]
    )

reporte_sunat = generar_reporte_calidad(df_silver_sunat, REGLAS_CALIDAD, "sunat_paises")
reporte_comtrade = generar_reporte_calidad(df_silver_comtrade, REGLAS_CALIDAD, "comtrade")
reporte_final = reporte_sunat.unionByName(reporte_comtrade)

reporte_final.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver.reporte_calidad")

display(reporte_final)

tabla,regla,total,cumplen,fallan,pct_fallan
sunat_paises,valor_no_negativo,2148,2148,0,0.0
sunat_paises,pais_valido,2148,2148,0,0.0
comtrade,valor_no_negativo,1,1,0,0.0
comtrade,pais_valido,1,1,0,0.0


In [0]:
%sql
-- 1. Sumamos todos los países de SUNAT para sacar el gran total de Perú
CREATE OR REPLACE TEMP VIEW gran_total_sunat AS
SELECT YEAR(fecha) AS anio, SUM(valor_fob_usd) AS total_sunat_macro
FROM silver.sunat_paises
GROUP BY YEAR(fecha);

-- 2. Comtrade ya viene sumado bajo el concepto "MUNDO"
CREATE OR REPLACE TEMP VIEW gran_total_comtrade AS
SELECT anio, SUM(valor_fob_usd) AS total_comtrade_cafe
FROM silver.comtrade
GROUP BY anio;

-- 3. Comparamos los miles de millones globales vs los millones del café
SELECT s.anio, 
       s.total_sunat_macro, 
       c.total_comtrade_cafe
FROM gran_total_sunat s
JOIN gran_total_comtrade c ON s.anio = c.anio;

anio,total_sunat_macro,total_comtrade_cafe
2005,1.730107631838E13,4.02822345E8
